# 🧩 Mini-Lab: Failure Mode Prediction

**Module: Is It Any Good? Quality, Latency, Cost** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Identify** common AI failure modes and explain why they're dangerous in production
2. **Recognize** silent failures — situations where the system returns a confident-looking but wrong answer without any error signal
3. **Implement** bounded retries to handle transient API failures without cascading into infinite loops

## Target Concepts

| Concept | Gloss |
|---------|-------|
| AI Failure Modes | Common ways AI apps fail in the real world: out-of-schema labels, data and user drift, rate limits, silent fallbacks, and input drift that degrade quality, latency, or cost without obvious errors. |
| Silent Failure | An empty filter result or coerced dtype can silently change what your code is operating on, producing confident, wrong output without any exception or traceback. |
| Bounded Retry | Retry transient errors only, and cap attempts, total wall-clock time, and cost or rate-limit budget so failures end fast, predictably, and safely. |


## Why Failure Modes Matter

Your AI system works great on 50 test examples. You ship it. Then it processes 10,000 real requests and things start going wrong — but **differently** from how a traditional program fails.

Traditional software crashes loudly: a `KeyError`, a database timeout, a 500 status code. You see the error and fix it.

AI systems often **fail silently**: they return a confident answer that happens to be wrong. No exception is raised. No log screams at you. The bad output flows downstream, and you only discover the problem when a customer complains — or worse, when you never notice at all.

This mini-lab walks through the most common failure modes so you can **predict** how your system will break *before* it ships.

## Step 1 — Catalog the Failure Modes

Let's define the common ways an LLM-based system can fail in production. We'll simulate each one so you can see what the symptoms look like.

In [1]:
from IPython.display import display, Markdown
def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

failure_modes = [
    {
        "name": "API Timeout / Rate Limit",
        "symptom": "Request hangs or returns HTTP 429/503",
        "danger": "Medium — you see an error, but if unhandled it crashes your pipeline",
        "silent": False,
    },
    {
        "name": "Model Refusal",
        "symptom": "Model returns 'I cannot help with that' instead of a classification",
        "danger": "High — the response is valid HTTP 200, but the content is useless",
        "silent": True,
    },
    {
        "name": "Format Violation",
        "symptom": "You expect a JSON object but get free-text or malformed JSON",
        "danger": "High — downstream parsing breaks or silently misreads the output",
        "silent": True,
    },
    {
        "name": "Hallucinated Label",
        "symptom": "Model invents a label not in your allowed set (e.g., 'refund' instead of 'billing')",
        "danger": "Very High — looks like a valid prediction, no error raised",
        "silent": True,
    },
    {
        "name": "Confident Wrong Answer",
        "symptom": "Model returns a valid label with high confidence, but it's the wrong one",
        "danger": "Very High — completely invisible without a gold set to check against",
        "silent": True,
    },
]

rows = ["| # | Failure Mode | Symptom | Silent? |", "|---|---|---|---|"]
for i, fm in enumerate(failure_modes, 1):
    silent_flag = "🔇 YES" if fm["silent"] else "🔊 No"
    rows.append(f"| {i} | **{fm['name']}** | {fm['symptom']} | {silent_flag} |")

md("### Common AI Failure Modes\n\n" + "\n".join(rows) + "\n\n"
   "> Notice how 4 out of 5 failure modes are **silent** — the system doesn't crash, "
   "it just gives you bad output that looks normal.")

### Common AI Failure Modes

| # | Failure Mode | Symptom | Silent? |
|---|---|---|---|
| 1 | **API Timeout / Rate Limit** | Request hangs or returns HTTP 429/503 | 🔊 No |
| 2 | **Model Refusal** | Model returns 'I cannot help with that' instead of a classification | 🔇 YES |
| 3 | **Format Violation** | You expect a JSON object but get free-text or malformed JSON | 🔇 YES |
| 4 | **Hallucinated Label** | Model invents a label not in your allowed set (e.g., 'refund' instead of 'billing') | 🔇 YES |
| 5 | **Confident Wrong Answer** | Model returns a valid label with high confidence, but it's the wrong one | 🔇 YES |

> Notice how 4 out of 5 failure modes are **silent** — the system doesn't crash, it just gives you bad output that looks normal.

## Step 2 — Simulate Silent Failures

Let's simulate a classification system that processes support tickets. Most calls work fine, but some fail silently — no exception, no error code, just a wrong or unusable answer.

In [2]:
import random
random.seed(42)

VALID_LABELS = {"billing", "technical", "account"}

def simulate_ai_response(ticket_text):
    """Simulate an LLM classification call that sometimes fails silently."""
    roll = random.random()

    if roll < 0.65:
        # Normal: returns a valid label
        return {"label": random.choice(list(VALID_LABELS)), "status": "success"}
    elif roll < 0.75:
        # Silent failure: hallucinated label
        return {"label": "refund", "status": "success"}  # not in VALID_LABELS!
    elif roll < 0.85:
        # Silent failure: model refusal disguised as an answer
        return {"label": "I'm sorry, I cannot classify this.", "status": "success"}
    elif roll < 0.92:
        # Silent failure: format violation (returned raw text instead of just a label)
        return {"label": "The category is billing.", "status": "success"}
    else:
        # Loud failure: API error (this one you'd notice)
        raise TimeoutError("API request timed out")


# Process 20 tickets and collect results
tickets = [f"Ticket #{i+1}" for i in range(20)]
results = []

for ticket in tickets:
    try:
        response = simulate_ai_response(ticket)
        results.append({"ticket": ticket, **response, "error": None})
    except Exception as e:
        results.append({"ticket": ticket, "label": None, "status": "error", "error": str(e)})

print(f"Processed {len(results)} tickets")
print(f"Errors caught: {sum(1 for r in results if r['status'] == 'error')}")
print(f"Successful responses: {sum(1 for r in results if r['status'] == 'success')}")

Processed 20 tickets
Errors caught: 0
Successful responses: 20


All the "successful" responses came back without any error. But are they all actually valid? Let's check.

In [3]:
# Detect silent failures: responses that look "successful" but aren't valid
loud_failures = []
silent_failures = []
clean_results = []

for r in results:
    if r["status"] == "error":
        loud_failures.append(r)
    elif r["label"] not in VALID_LABELS:
        silent_failures.append(r)
    else:
        clean_results.append(r)

print(f"Clean results:    {len(clean_results)}")
print(f"Loud failures:    {len(loud_failures)}  (you saw these — exceptions were raised)")
print(f"Silent failures:  {len(silent_failures)}  (NO error raised — looked fine!)")

print("\n--- Silent failures (these slipped through!) ---")
for sf in silent_failures:
    print(f"  {sf['ticket']}: label = {sf['label']!r}")

Clean results:    14
Loud failures:    0  (you saw these — exceptions were raised)
Silent failures:  6  (NO error raised — looked fine!)

--- Silent failures (these slipped through!) ---
  Ticket #2: label = 'refund'
  Ticket #4: label = 'refund'
  Ticket #5: label = 'refund'
  Ticket #6: label = 'The category is billing.'
  Ticket #12: label = 'refund'
  Ticket #15: label = "I'm sorry, I cannot classify this."


**This is the core danger:** the system returned HTTP 200, no exception was raised, and the response *looks* like it worked. But labels like `"refund"`, `"I'm sorry, I cannot classify this."`, and `"The category is billing."` are all garbage.

Without a **validation check** against your known label set, these silent failures flow straight into your database, your dashboard, or your customer's experience.

## Step 3 — Bounded Retries for Loud Failures

For **loud failures** (timeouts, rate limits), the fix is a **bounded retry**: retry the call a fixed number of times with increasing delays, then give up gracefully. The key word is *bounded* — if you retry forever, one slow API brings down your entire pipeline.

In [4]:
import time

def unreliable_api_call(ticket):
    """Simulate an API that fails ~50% of the time with timeouts."""
    if random.random() < 0.5:
        raise TimeoutError("API request timed out")
    return {"label": "billing", "status": "success"}


def call_with_retry(ticket, max_retries=3, base_delay=0.1):
    """
    Call the API with bounded retries.
    - max_retries: how many EXTRA attempts after the first failure
    - base_delay: initial wait in seconds (doubles each retry)
    """
    for attempt in range(max_retries + 1):
        try:
            print(f"  Attempt {attempt + 1}/{max_retries + 1} for '{ticket}'...")
            result = unreliable_api_call(ticket)
            return {"ticket": ticket, **result, "error": None}
        except TimeoutError:
            if attempt < max_retries:
                wait = base_delay * (2 ** attempt)  # exponential backoff
                print(f"    ⏳ Timeout. Retrying in {wait:.1f}s...")
                time.sleep(wait)
            else:
                print(f"    ❌ All {max_retries + 1} attempts failed. Giving up.")
                return {"ticket": ticket, "label": None, "status": "failed", "error": "max retries exceeded"}


# Demo: process 5 tickets with bounded retries
random.seed(7)
for i in range(5):
    result = call_with_retry(f"Ticket #{i+1}", max_retries=3, base_delay=0.05)
    status_icon = "✅" if result["status"] == "success" else "❌"
    print(f"  {status_icon} Result: {result['status']}\n")

  Attempt 1/4 for 'Ticket #1'...
    ⏳ Timeout. Retrying in 0.1s...
  Attempt 2/4 for 'Ticket #1'...
    ⏳ Timeout. Retrying in 0.1s...
  Attempt 3/4 for 'Ticket #1'...
  ✅ Result: success

  Attempt 1/4 for 'Ticket #2'...
    ⏳ Timeout. Retrying in 0.1s...


  Attempt 2/4 for 'Ticket #2'...
  ✅ Result: success

  Attempt 1/4 for 'Ticket #3'...
    ⏳ Timeout. Retrying in 0.1s...
  Attempt 2/4 for 'Ticket #3'...
    ⏳ Timeout. Retrying in 0.1s...
  Attempt 3/4 for 'Ticket #3'...
  ✅ Result: success

  Attempt 1/4 for 'Ticket #4'...
    ⏳ Timeout. Retrying in 0.1s...
  Attempt 2/4 for 'Ticket #4'...
    ⏳ Timeout. Retrying in 0.1s...


  Attempt 3/4 for 'Ticket #4'...
    ⏳ Timeout. Retrying in 0.2s...
  Attempt 4/4 for 'Ticket #4'...
    ❌ All 4 attempts failed. Giving up.
  ❌ Result: failed

  Attempt 1/4 for 'Ticket #5'...
    ⏳ Timeout. Retrying in 0.1s...


  Attempt 2/4 for 'Ticket #5'...
  ✅ Result: success



**Key points about bounded retries:**
- The retry count is **fixed** — the loop always terminates
- **Exponential backoff** (doubling the delay) avoids hammering a struggling API
- After all retries fail, the system returns a **graceful failure** (a record with `status: failed`) instead of crashing
- Bounded retries handle **loud** failures; they do nothing for **silent** failures

## Step 4 — Predict Your Failure Modes

Before shipping, fill out a **failure mode prediction table** — a checklist that forces you to think through how each failure will manifest, whether you'll notice it, and what your mitigation plan is.

In [5]:
failure_prediction = """
### 📋 Failure Mode Prediction Table

| Failure Mode | Will I notice? | Symptom | Mitigation |
|---|---|---|---|
| **API Timeout** | ✅ Yes — exception raised | Request hangs, pipeline crashes | Bounded retry with exponential backoff (max 3 retries) |
| **Rate Limit (429)** | ✅ Yes — HTTP error code | Burst of errors in logs | Bounded retry + request throttling |
| **Model Refusal** | ❌ No — HTTP 200 returned | Label field contains refusal text | Validate response against allowed label set |
| **Hallucinated Label** | ❌ No — HTTP 200 returned | Label not in allowed set | Validate response against allowed label set |
| **Format Violation** | ❌ Maybe — depends on parsing | JSON parse error or wrong field | Validate expected schema before using the result |
| **Confident Wrong Answer** | ❌ No — looks perfect | Correct format, wrong label | Only detectable by scoring against a gold set periodically |

> **Pattern:** Loud failures (timeouts, rate limits) are easy — bounded retries handle them.  
> Silent failures (refusals, hallucinations, wrong answers) are dangerous — you need **validation checks** and **periodic gold-set scoring** to catch them.
"""

md(failure_prediction)


### 📋 Failure Mode Prediction Table

| Failure Mode | Will I notice? | Symptom | Mitigation |
|---|---|---|---|
| **API Timeout** | ✅ Yes — exception raised | Request hangs, pipeline crashes | Bounded retry with exponential backoff (max 3 retries) |
| **Rate Limit (429)** | ✅ Yes — HTTP error code | Burst of errors in logs | Bounded retry + request throttling |
| **Model Refusal** | ❌ No — HTTP 200 returned | Label field contains refusal text | Validate response against allowed label set |
| **Hallucinated Label** | ❌ No — HTTP 200 returned | Label not in allowed set | Validate response against allowed label set |
| **Format Violation** | ❌ Maybe — depends on parsing | JSON parse error or wrong field | Validate expected schema before using the result |
| **Confident Wrong Answer** | ❌ No — looks perfect | Correct format, wrong label | Only detectable by scoring against a gold set periodically |

> **Pattern:** Loud failures (timeouts, rate limits) are easy — bounded retries handle them.  
> Silent failures (refusals, hallucinations, wrong answers) are dangerous — you need **validation checks** and **periodic gold-set scoring** to catch them.


## 🎯 Summary

### Key Takeaways

1. **AI failure modes** — AI systems break differently from traditional software; the most common failures include timeouts, refusals, format violations, hallucinated labels, and confident wrong answers
2. **Silent failures are the biggest risk** — 4 out of 5 common failure modes return HTTP 200 with no error raised; without validation against an allowed label set or periodic gold-set checks, bad outputs flow downstream unnoticed
3. **Bounded retry** — retry transient API failures a fixed number of times with exponential backoff, then fail gracefully; never retry forever
4. **Predict before you ship** — fill out a failure mode prediction table to anticipate how each failure will manifest, whether you'll notice it, and what your mitigation plan is